In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "notebooks" else HERE
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [2]:
DATA_PATH = RAW_DIR / "file.csv"
assert DATA_PATH.exists(), f"Missing raw export: {DATA_PATH}"
df_raw = pd.read_csv(DATA_PATH, encoding="utf-16", sep="\t", low_memory=False)
print("Raw shape:", df_raw.shape)

Raw shape: (122419, 33)


# Part 2  Data Cleaning, Validation and Reconciliation

After reviewing the original dataset, I cleaned and organised the records
so they could be used reliably in the analysis.

The main focus was to separate actual sales from refunds and voided
transactions, check the financial values, and prepare daily sales figures.

I also kept track of any assumptions made during cleaning to avoid
misrepresenting the business's sales.

## 2.1 Preserve source and convert types


In [3]:
df = df_raw.copy()
df["sale_date"] = pd.to_datetime(df["Date"], errors="coerce").dt.normalize()
df["sale_time"] = pd.to_datetime(df["Time"].astype("string"), format="%H:%M:%S", errors="coerce").dt.time
money_cols = ["Product Sales", "Discounts", "Net Sales", "Tax", "Gross Sales"]
for col in money_cols:
    cleaned_text = (df[col].astype("string")
                    .str.replace("€", "", regex=False)
                    .str.replace(",", "", regex=False)
                    .str.strip())
    df[col + "_eur"] = pd.to_numeric(cleaned_text, errors="coerce")

conversion_audit = pd.DataFrame({
    "field": money_cols,
    "unparseable": [int(df[c + "_eur"].isna().sum()) for c in money_cols],
    "negative": [int(df[c + "_eur"].lt(0).sum()) for c in money_cols],
    "zero": [int(df[c + "_eur"].eq(0).sum()) for c in money_cols]
})
display(conversion_audit)
print("Invalid dates:", df["sale_date"].isna().sum(), "| Invalid times:", df["sale_time"].isna().sum())
assert df["sale_date"].notna().all(), "Review invalid dates before proceeding"
assert df[[c + "_eur" for c in money_cols]].notna().all().all(), "Review unparseable money fields"

,field,unparseable,negative,zero
0,Product Sales,0,844,15419
1,Discounts,0,0,122419
2,Net Sales,0,844,15419
3,Tax,0,1,103687
4,Gross Sales,0,844,15419


Invalid dates: 0 | Invalid times: 0


## 2.2 Identify dates without POS records

In [4]:
calendar = pd.date_range(df["sale_date"].min(), df["sale_date"].max(), freq="D")
missing_dates = calendar.difference(pd.DatetimeIndex(df["sale_date"].dropna().unique()))
print(f"Missing dates ({len(missing_dates)}):", [x.strftime("%Y-%m-%d") for x in missing_dates])

Missing dates (5): ['2024-12-25', '2024-12-26', '2025-01-01', '2025-01-02', '2025-01-24']


## 2.3 Classify records before filtering
Identify voids by item label and refunds by event type. Keep overlapping cases visible. Zero-priced items are **not automatically voids**.

In [5]:
df["event_type_clean"] = df["Event Type"].astype("string").str.strip().str.casefold()
df["is_refund"] = df["event_type_clean"].eq("refund").fillna(False)
df["is_payment"] = df["event_type_clean"].eq("payment").fillna(False)
df["is_void"] = df["Item"].astype("string").str.contains(r"\(Voided\)", case=False, regex=True, na=False)
df["is_exact_duplicate"] = df_raw.duplicated(keep=False).to_numpy()
df["record_group"] = np.select(
    [df["is_refund"] & df["is_void"], df["is_refund"], df["is_void"], df["is_payment"]],
    ["refund + void label", "refund", "void label", "payment"], default="other")
summary = df.groupby("record_group", dropna=False).agg(
    rows=("Transaction ID", "size"),
    transaction_ids=("Transaction ID", "nunique"),
    quantity=("Qty", "sum"),
    gross_eur=("Gross Sales_eur", "sum"),
    net_eur=("Net Sales_eur", "sum")
).reset_index()
display(summary)
print("Exact duplicate occurrences (excluding first):", df_raw.duplicated().sum())
print("Rows belonging to duplicate groups:", df["is_exact_duplicate"].sum())

,record_group,rows,transaction_ids,quantity,gross_eur,net_eur
0,payment,106156,58891,134013.0,548244.35,536021.5
1,refund,844,342,-1218.0,-5568.0,-5554.63
2,void label,15419,271,19796.0,0.0,0.0


Exact duplicate occurrences (excluding first): 8985
Rows belonging to duplicate groups: 12788


The dataset contains payments, refunds and voided items, so I separated
these before calculating revenue.

Voided items were excluded from sales, while refunds were kept separately
to calculate the final revenue after refunds.

I did not automatically remove zero-priced items or duplicate rows,
as these could represent genuine transactions.

## 2.4 Investigate duplicates and monetary consistency
Identical exported lines may be multiple units sold. 

In [6]:
dup_examples = (df.loc[df["is_exact_duplicate"],
                       ["sale_date", "Time", "Transaction ID", "Item", "Qty", "Gross Sales_eur", "Event Type"]]
                .head(15))
display(dup_examples)
# Accounting check: gross = net + tax (may differ for some Square export conventions)
df["gross_check_difference"] = (df["Gross Sales_eur"] - df["Net Sales_eur"] - df["Tax_eur"]).round(2)
display(df["gross_check_difference"].value_counts().head(10).rename_axis("difference_eur").to_frame("rows"))
print("Rows with > €0.01 gross/net/tax difference:", df["gross_check_difference"].abs().gt(.01).sum())

,sale_date,Time,Transaction ID,Item,Qty,Gross Sales_eur,Event Type
15,2025-11-30,20:44:27,UdHVPtzbpOQQOQc6T8ej9RyjlUXZY,Cans Drink,1.0,1.65,Payment
17,2025-11-30,20:44:27,UdHVPtzbpOQQOQc6T8ej9RyjlUXZY,Cans Drink,1.0,1.65,Payment
65,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Gelato (Voided),2.0,0.0,Payment
66,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Topping (Voided),1.0,0.0,Payment
70,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Cadbury Hot Chocolate (Voided),1.0,0.0,Payment
71,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Americano (Voided),1.0,0.0,Payment
72,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Gelato (Voided),3.0,0.0,Payment
73,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Gelato (Voided),1.0,0.0,Payment
74,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Topping (Voided),5.0,0.0,Payment
75,2025-11-30,19:00:11,C2QLQFxTflqAciYBRtTZ9qbHev5YY,Latte (Voided),1.0,0.0,Payment


,rows
difference_eur,
0.0,122419


Rows with > €0.01 gross/net/tax difference: 0


## 2.5 Inspect missing business fields without inventing values


In [7]:
df["category_analysis"] = df["Category"].astype("string").str.strip().fillna("Uncategorised")
df.loc[df["category_analysis"].eq(""), "category_analysis"] = "Uncategorised"
df["item_analysis"] = df["Item"].astype("string").str.strip()
print("Uncategorised records:", df["category_analysis"].eq("Uncategorised").sum())
print("Records with missing dining option:", df["Dining Option"].isna().sum())
display(df.loc[df["category_analysis"].eq("Uncategorised"), "Item"].value_counts().head(15).to_frame("rows"))

Uncategorised records: 3790
Records with missing dining option: 26897


,rows
Item,
Red bull Sha,1243
Custom Amount,390
Black pudding sausage roll,366
Red bull Sha (Voided),265
Ham & Cheese Crep,245
Vani Late,227
Scone,215
Cadbury Hot Choco Whit Mashmakllow,111
Custom Amount (Voided),92


## 2.6 Build explicit analysis populations
**Sales-only:** payment events excluding void-labelled lines. **Refunds:** kept separately for financial reconciliation. **All economic events:** sales and refunds, excluding void-labelled lines. No exact duplicate lines are dropped. These are working definitions to validate against Square and Power BI.

In [8]:
sales_lines = df.loc[df["is_payment"] & ~df["is_void"]].copy()
refund_lines = df.loc[df["is_refund"] & ~df["is_void"]].copy()
economic_lines = df.loc[(df["is_payment"] | df["is_refund"]) & ~df["is_void"]].copy()
for name, data in [("Raw export", df), ("Sales-only", sales_lines),
                   ("Refunds", refund_lines), ("Economic events", economic_lines)]:
    print(f"{name:17} rows={len(data):>7,}  transactions={data['Transaction ID'].nunique():>7,}  "
          f"gross=€{data['Gross Sales_eur'].sum():>12,.2f}  net=€{data['Net Sales_eur'].sum():>12,.2f}  qty={data['Qty'].sum():>10,.0f}")
print("Refund amount sign sample:")
display(refund_lines[["Qty", "Gross Sales_eur", "Net Sales_eur"]].describe())

Raw export        rows=122,419  transactions= 59,238  gross=€  542,676.35  net=€  530,466.87  qty=   152,591
Sales-only        rows=106,156  transactions= 58,891  gross=€  548,244.35  net=€  536,021.50  qty=   134,013
Refunds           rows=    844  transactions=    342  gross=€   -5,568.00  net=€   -5,554.63  qty=    -1,218
Economic events   rows=107,000  transactions= 59,233  gross=€  542,676.35  net=€  530,466.87  qty=   132,795
Refund amount sign sample:


,Qty,Gross Sales_eur,Net Sales_eur
count,844.000000,844.0,844.0
mean,-1.443128,-6.597156,-6.581315
std,1.108722,5.080074,4.964092
min,-16.000000,-60.0,-60.0
25%,-2.000000,-7.5,-7.5
50%,-1.000000,-6.0,-6.0
75%,-1.000000,-3.8,-3.8
max,-1.000000,-0.5,-0.5


## 2.7 Prepare daily data for later EDA and forecasting

### Note: Public Holidays and Missing Trading Dates

The dataset covers 1 December 2024 to 30 November 2025, with sales recorded on 360 days.

No transactions were recorded on the following dates:
- **25 December 2024** – Christmas Day
- **26 December 2024** – St. Stephen's Day
- **1 January 2025** – New Year's Day
- **2 January 2025** - New Years's Holiday
- **24 January 2025**

These dates were excluded from the observed-day sales analysis rather than treated as zero-sales days, since the available data does not confirm whether the business was closed.


In [10]:
daily_sales = (sales_lines.groupby("sale_date", as_index=False)
    .agg(gross_revenue_eur=("Gross Sales_eur", "sum"),
         net_sales_eur=("Net Sales_eur", "sum"),
         units_sold=("Qty", "sum"),
         transactions=("Transaction ID", "nunique"))
    .sort_values("sale_date"))
daily_sales["weekday"] = daily_sales["sale_date"].dt.day_name()
daily_sales["month"] = daily_sales["sale_date"].dt.month
print("Observed days:", len(daily_sales))
display(daily_sales.head())
assert len(daily_sales) == sales_lines["sale_date"].nunique()
assert np.isclose(daily_sales["gross_revenue_eur"].sum(), sales_lines["Gross Sales_eur"].sum())

Observed days: 360


,sale_date,gross_revenue_eur,net_sales_eur,units_sold,transactions,weekday,month
0,2024-12-01,1354.8,1354.8,345.0,137,Sunday,12
1,2024-12-02,674.4,674.4,172.0,93,Monday,12
2,2024-12-03,878.1,878.1,215.0,102,Tuesday,12
3,2024-12-04,854.8,854.8,242.0,111,Wednesday,12
4,2024-12-05,832.6,832.6,210.0,100,Thursday,12


## 2.8 Final cleaning audit and optional exports


In [ ]:
cleaning_audit = pd.DataFrame([
    ("Original rows", len(df_raw)),
    ("Exact duplicate occurrences retained", int(df_raw.duplicated().sum())),
    ("Refund rows (all)", int(df["is_refund"].sum())),
    ("Void-labelled rows (all)", int(df["is_void"].sum())),
    ("Sales-only rows", len(sales_lines)),
    ("Refund rows excluding void-labelled", len(refund_lines)),
    ("Economic-event rows", len(economic_lines)),
    ("Dates without records", len(missing_dates)),
], columns=["metric", "value"])
display(cleaning_audit)
# daily_sales.to_csv("melt_daily_sales_clean.csv", index=False)
print("Part 2 complete. Next: descriptive statistics, EDA, and category-level demand.")

,metric,value
0,Original rows,122419
1,Exact duplicate occurrences retained,8985
2,Refund rows (all),844
3,Void-labelled rows (all),15419
4,Sales-only rows,106156
5,Refund rows excluding void-labelled,844
6,Economic-event rows,107000
7,Dates without records,5


Part 2 complete. Next: descriptive statistics, EDA, and category-level demand.


### Key Observations

The cleaning process identified 106,156 valid payment item records,
844 refund records and 15,419 void-labelled records.

Payment transactions generated €548,244.35 in gross revenue.
After accounting for refunds, the gross revenue was €542,676.35.

The financial values passed the reconciliation checks, and duplicate
records were retained rather than removed without sufficient evidence.

This provided a consistent starting point for the sales analysis,
while keeping payment-only and refund-adjusted figures separate.

## 2.9 Improve missing-category labels (conservative mapping)


In [12]:
known = df.loc[df["Category"].notna(), ["item_analysis", "Category"]].copy()
known["Category"] = known["Category"].astype("string").str.strip()
known = known.loc[known["Category"].ne("") & known["item_analysis"].notna()]
unique_category_counts = known.groupby("item_analysis")["Category"].nunique()
unambiguous_items = unique_category_counts[unique_category_counts.eq(1)].index
category_lookup = (known.loc[known["item_analysis"].isin(unambiguous_items)]
                   .drop_duplicates("item_analysis").set_index("item_analysis")["Category"])
missing_before = df["category_analysis"].eq("Uncategorised").sum()
proposed = df["item_analysis"].map(category_lookup)
mask = df["category_analysis"].eq("Uncategorised") & proposed.notna()
df.loc[mask, "category_analysis"] = proposed.loc[mask]
print(f"Missing before: {missing_before:,} | inferred: {mask.sum():,} | unresolved: {df['category_analysis'].eq('Uncategorised').sum():,}")
display(df.loc[mask, ["item_analysis", "category_analysis"]].drop_duplicates().head(20))
# Refresh derived analysis populations after category updates
sales_lines = df.loc[df["is_payment"] & ~df["is_void"]].copy()
refund_lines = df.loc[df["is_refund"] & ~df["is_void"]].copy()
economic_lines = df.loc[(df["is_payment"] | df["is_refund"]) & ~df["is_void"]].copy()

Missing before: 3,790 | inferred: 817 | unresolved: 2,973


,item_analysis,category_analysis
4698,Toping,Toppings
83631,Spicy Cajun wrap,Menu
83681,Ribena Juice,Menu
83689,Vani Late,Menu
83910,Black pudding sausage roll,Menu
84518,Choco Dubai Tub,Menu
85237,Plain Croissant,Menu
86129,Cheesecake Small Slice,Menu
96623,Nutella muffin,Menu
102067,Ballymalloe Sausage Sandwich,Menu


## 2.10 Construct refund-adjusted daily financial measures
Keep **payment-only sales** and **refund-adjusted revenue** as distinct columns. Refunds are assigned to the date of their recorded refund event, not retrospectively to the original purchase date.

In [13]:
daily_payments = (sales_lines.groupby("sale_date", as_index=False)
    .agg(payment_gross_eur=("Gross Sales_eur", "sum"),
         payment_net_eur=("Net Sales_eur", "sum"),
         units_sold=("Qty", "sum"),
         transactions=("Transaction ID", "nunique")))
daily_refunds = (refund_lines.groupby("sale_date", as_index=False)
    .agg(refund_gross_eur=("Gross Sales_eur", "sum"),
         refund_net_eur=("Net Sales_eur", "sum")))
daily_sales = daily_payments.merge(daily_refunds, on="sale_date", how="outer")
for col in ["payment_gross_eur", "payment_net_eur", "units_sold", "transactions", "refund_gross_eur", "refund_net_eur"]:
    daily_sales[col] = daily_sales[col].fillna(0)
daily_sales["refund_adjusted_gross_eur"] = daily_sales["payment_gross_eur"] + daily_sales["refund_gross_eur"]
daily_sales["refund_adjusted_net_eur"] = daily_sales["payment_net_eur"] + daily_sales["refund_net_eur"]
daily_sales = daily_sales.sort_values("sale_date").reset_index(drop=True)
daily_sales["weekday"] = daily_sales["sale_date"].dt.day_name()
daily_sales["month"] = daily_sales["sale_date"].dt.month
assert np.isclose(daily_sales["refund_adjusted_gross_eur"].sum(), economic_lines["Gross Sales_eur"].sum())
assert np.isclose(daily_sales["payment_gross_eur"].sum(), sales_lines["Gross Sales_eur"].sum())
print("Daily records:", len(daily_sales))
print("Refund-adjusted gross revenue (€):", round(daily_sales["refund_adjusted_gross_eur"].sum(), 2))
display(daily_sales.head())

Daily records: 360
Refund-adjusted gross revenue (€): 542676.35


,sale_date,payment_gross_eur,payment_net_eur,units_sold,transactions,refund_gross_eur,refund_net_eur,refund_adjusted_gross_eur,refund_adjusted_net_eur,weekday,month
0,2024-12-01,1354.8,1354.8,345.0,137,0.0,0.0,1354.8,1354.8,Sunday,12
1,2024-12-02,674.4,674.4,172.0,93,0.0,0.0,674.4,674.4,Monday,12
2,2024-12-03,878.1,878.1,215.0,102,0.0,0.0,878.1,878.1,Tuesday,12
3,2024-12-04,854.8,854.8,242.0,111,0.0,0.0,854.8,854.8,Wednesday,12
4,2024-12-05,832.6,832.6,210.0,100,0.0,0.0,832.6,832.6,Thursday,12


## 2.11 Export reproducible datasets
- The saved `cleaned_sales.csv` retains classified item-level records, including refund/void flags. 
- `daily_sales.csv` retains observed days and refund-adjusted gross revenue. Missing calendar days are **not** filled with zeros.

In [14]:
df.to_csv(PROCESSED_DIR / "cleaned_sales.csv", index=False)
daily_sales.to_csv(PROCESSED_DIR / "daily_sales.csv", index=False)
print("Saved cleaned_sales.csv and daily_sales.csv")

Saved cleaned_sales.csv and daily_sales.csv
